In [0]:
landing_table=dbutils.widgets.get("landing_table")
landing_flattened_table=dbutils.widgets.get("landing_flattened_table")
raw_table=dbutils.widgets.get("raw_table")
volume_path=dbutils.widgets.get("volume_path")

In [0]:
try:
    spark.sql(f"""
        COPY INTO {landing_table}
        FROM (
            SELECT
                CASE
                    WHEN CAST(AdjAmounts AS STRING) LIKE '[%' THEN from_json(CAST(AdjAmounts AS STRING), 'array<string>')
                    WHEN AdjAmounts IS NOT NULL THEN array(CAST(AdjAmounts AS STRING))
                    ELSE CAST(NULL AS array<string>)
                END AS AdjAmounts,
                CAST(RemitClaimID AS STRING) AS RemitClaimID,
                * EXCEPT (AdjAmounts, RemitClaimID),
                CAST(_metadata.file_name AS STRING) AS _file_name,
                CURRENT_TIMESTAMP() AS _load_timestamp
            FROM '{volume_path}'
        )
        FILEFORMAT = JSON
        COPY_OPTIONS ('mergeSchema' = 'true');
    """)
except Exception as e:
    print(f"Error loading into {landing_table}: {e}")
    raise 